# Case study, step 1: nearest neighbours from the self-trained models

Code for the case study in *When Validation Disagrees: How Validation Choices Shape Word Embedding Selection in Computational Communication Science*.

This notebook extracts the 100 nearest neighbours of the six keywords (Frau, Femizid, Mann, Mord, Opfer, Täter) from the 32 selected self-trained models (one per model family, see `06_casestudy/selected_models.csv`).

**Requires the trained models in `models/oembeddings/`, which are not shared** (> 1 TB). The output of this notebook is included in the repository, so the rest of the case study can be reproduced without the models.

**Output** (written to `06_casestudy/`): `table_allneighbours_{Keyword}.csv` with the columns `Model`, `Neighbor`, `Similarity`. Neighbours are lowercased so cased and lowercased models can be compared.

**Next steps** (all in `06_casestudy/`, see the README there):
1. `00_download_facebook_model.ipynb`: download cc.de.300
2. `01_extract_neighbours_facebook.ipynb`: add the cc.de.300 neighbours to the tables
3. `02_lemmatize_overlap.ipynb`: manual lemmatization, overlap statistics, heatmaps (Figures 8 and 9)

In [ ]:
from pathlib import Path
import gensim
from gensim.models.keyedvectors import KeyedVectors
import pandas as pd

In [ ]:
def load_model(file_path: Path) -> KeyedVectors:
    """Loads a word embedding model from the given file path."""
    file_path = str(file_path)
    if file_path.endswith('.bin'):
        model = gensim.models.fasttext.load_facebook_vectors(file_path)
    elif file_path.endswith('.vec'):
        model = gensim.models.KeyedVectors.load_word2vec_format(file_path, binary=False)
    else:
        raise ValueError(f"Unsupported file format: {file_path}")
    return model

def get_nearest_neighbors(model: KeyedVectors, keyword: str, top_n: int = 100) -> list:
    """Fetches the top N nearest neighbors for a given keyword from the model."""
    return model.most_similar(keyword, topn=top_n)

In [ ]:
# Keywords: lowercase variant for lowercased models, cased variant for cased models
keywords = {
    'frau':    ('frau', 'Frau'),
    'femizid': ('femizid', 'Femizid'),
    'täter':   ('täter', 'Täter'),
    'opfer':   ('opfer', 'Opfer'),
    'mann':    ('mann', 'Mann'),
    'mord':    ('mord', 'Mord'),
}

# Repository root (works when run from the repo root or from 05_analyse/)
p = Path.cwd()
if "05_analyse" in p.name:
    p = p.parent

model_dir = p / 'models' / 'oembeddings'
assert model_dir.exists(), "Could not find directory with models. Please check pathing"

# All case-study outputs go to 06_casestudy/
out_dir = p / '06_casestudy'
out_dir.mkdir(exist_ok=True)

model_files = sorted(model_dir.glob("*.vec"))
print(f"Found {len(model_files)} models in {model_dir}")

In [ ]:
for keyword, (keyword_lower, keyword_cased) in keywords.items():
    all_neighbors = []

    for model_file in model_files:
        model_name = model_file.stem
        print(f"Processing model: {model_name} for keyword: {keyword}")

        selected_keyword = keyword_lower if 'lower' in model_name else keyword_cased

        model = load_model(model_file)
        try:
            neighbors = get_nearest_neighbors(model, selected_keyword)
        except KeyError:
            print(f"  '{selected_keyword}' not in vocabulary of {model_name}, skipping")
            continue

        for neighbor, similarity in neighbors:
            all_neighbors.append({'Model': model_name, 'Neighbor': neighbor.lower(), 'Similarity': similarity})

    out_file = out_dir / f'table_allneighbours_{keyword_cased}.csv'
    df = pd.DataFrame(all_neighbors)
    df.to_csv(out_file, index=False)
    print(f"[{keyword_cased}] {df['Model'].nunique()} models, {df['Neighbor'].nunique()} unique neighbours -> {out_file}")